# 🛒 Superstore Sales Analysis
### Addressing Key Business Questions

## Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

data = pd.read_csv("superstore.csv", encoding="latin-1")
print("Dataset loaded! Shape:", data.shape)

## Basic Data Checks

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.describe()

---
## Q1: Which Products Are the Top Sellers?

In [ ]:
# Top 5 products by total sales
data.groupby('product_name')['sales'].sum().sort_values(ascending=False).head()

In [ ]:
# Top 10 — visualised
top_products = data.groupby('product_name')['sales'].sum() \
                   .sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
top_products.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Products by Sales')
plt.xlabel('Total Sales ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 products by quantity sold
data.groupby('product_name')['quantity'].sum().sort_values(ascending=False).head()

In [ ]:
# Sales by Sub-Category
data.groupby('sub_category')['sales'].sum().sort_values(ascending=False)

In [ ]:
# Sub-Category sales bar chart
subcat_sales = data.groupby('sub_category')['sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
subcat_sales.plot(kind='bar', color='coral')
plt.title('Sales by Sub-Category')
plt.xlabel('Sub-Category')
plt.ylabel('Sales ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Q2: What Are the Most Profitable Regions and Customer Segments?

In [ ]:
# Top regions by profit
data.groupby('region')['profit'].sum().sort_values(ascending=False).head()

In [ ]:
# Region — Sales & Profit together
region_summary = data.groupby('region').agg(
    Total_Sales  = ('sales',  'sum'),
    Total_Profit = ('profit', 'sum'),
    Num_Orders   = ('order_id', 'count')
).round(2).sort_values('Total_Profit', ascending=False)

region_summary

In [ ]:
# Region profit bar chart
region_summary['Total_Profit'].plot(kind='bar', color='mediumseagreen')
plt.title('Total Profit by Region')
plt.xlabel('Region')
plt.ylabel('Profit ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Most profitable customer segments
data.groupby('segment')['profit'].sum().sort_values(ascending=False).head()

In [ ]:
# Segment — full summary
segment_summary = data.groupby('segment').agg(
    Total_Sales   = ('sales',  'sum'),
    Total_Profit  = ('profit', 'sum'),
    Avg_Order_Val = ('sales',  'mean'),
    Num_Customers = ('customer_id', 'nunique')
).round(2).sort_values('Total_Profit', ascending=False)

segment_summary

In [ ]:
# Segment profit pie chart
data.groupby('segment')['profit'].sum().plot(
    kind='pie', autopct='%1.1f%%', startangle=90,
    colors=['#4C72B0','#DD8452','#55A868'])
plt.title('Profit Share by Segment')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Region × Segment (multi-level groupby)
reg_seg = data.groupby(['region','segment'])['profit'].sum().unstack()
reg_seg

In [ ]:
# Heatmap — Region vs Segment profit
plt.figure(figsize=(8, 5))
sns.heatmap(reg_seg, annot=True, fmt='.0f', cmap='YlGnBu', linewidths=0.5)
plt.title('Profit Heatmap: Region × Segment')
plt.tight_layout()
plt.show()

---
## Q3: How Do Seasonal Trends Affect Overall Sales?

In [ ]:
# Parse dates
data['order_date'] = pd.to_datetime(data['order_date'], dayfirst=True)
data['Year']       = data['order_date'].dt.year
data['Month']      = data['order_date'].dt.month
data['Month_Name'] = data['order_date'].dt.strftime('%b')
data['Quarter']    = data['order_date'].dt.quarter
print("Date columns created!")

In [ ]:
# Monthly sales trend
monthly = data.groupby(['Year','Month'])['sales'].sum().reset_index()
monthly['Period'] = pd.to_datetime(monthly[['Year','Month']].assign(day=1))

plt.figure(figsize=(12, 5))
plt.plot(monthly['Period'], monthly['sales'], marker='o', color='steelblue')
plt.title('Monthly Sales Trend')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Average sales by month (seasonality pattern)
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
monthly_avg = data.groupby('Month_Name')['sales'].mean().reindex(month_order)
monthly_avg

In [ ]:
# Seasonality bar chart
monthly_avg.plot(kind='bar', color='mediumpurple')
plt.title('Average Sales by Month (Seasonality)')
plt.xlabel('Month')
plt.ylabel('Avg Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Quarterly sales summary
quarterly = data.groupby(['Year','Quarter'])['sales'].sum().unstack()
quarterly

In [ ]:
# Quarterly trend chart
quarterly.plot(kind='bar', colormap='Set2')
plt.title('Quarterly Sales by Year')
plt.xlabel('Year')
plt.ylabel('Sales ($)')
plt.xticks(rotation=0)
plt.legend(title='Quarter', labels=['Q1','Q2','Q3','Q4'])
plt.tight_layout()
plt.show()

---
## Q4: Underlying Factors — Pricing, Promotions (Discounts), Inventory

In [ ]:
# Discount vs Profit scatter
plt.figure()
sns.scatterplot(data=data, x='discount', y='profit', alpha=0.4, color='tomato')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Effect of Discount on Profit')
plt.xlabel('Discount Rate')
plt.ylabel('Profit ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Average profit at each discount level
data.groupby('discount')['profit'].mean().sort_index()

In [ ]:
# Avg profit by discount — bar chart
disc_profit = data.groupby('discount')['profit'].mean()

disc_profit.plot(kind='bar', color=['green' if v >= 0 else 'red' for v in disc_profit])
plt.title('Average Profit by Discount Level')
plt.xlabel('Discount')
plt.ylabel('Avg Profit ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Sales vs Profit by Category
cat_summary = data.groupby('category').agg(
    Total_Sales  = ('sales',  'sum'),
    Total_Profit = ('profit', 'sum'),
    Avg_Discount = ('discount','mean'),
    Total_Qty    = ('quantity','sum')
).round(2)

cat_summary

In [ ]:
# Correlation heatmap — key numeric variables
plt.figure(figsize=(7, 5))
num_cols = data[['sales','quantity','discount','profit']]
sns.heatmap(num_cols.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation: Sales, Quantity, Discount, Profit')
plt.tight_layout()
plt.show()

In [ ]:
# Sales vs Profit scatter coloured by Category
plt.figure()
sns.scatterplot(data=data, x='sales', y='profit', hue='category', alpha=0.5)
plt.title('Sales vs Profit by Category')
plt.xlabel('Sales ($)')
plt.ylabel('Profit ($)')
plt.tight_layout()
plt.show()

---
## Final Summary — Key Metrics

In [ ]:
total_sales    = data['sales'].sum()
total_profit   = data['profit'].sum()
margin         = total_profit / total_sales * 100
total_orders   = data['order_id'].nunique()
total_customers= data['customer_id'].nunique()
best_product   = data.groupby('product_name')['sales'].sum().idxmax()
best_region    = data.groupby('region')['profit'].sum().idxmax()
best_segment   = data.groupby('segment')['profit'].sum().idxmax()
worst_subcat   = data.groupby('sub_category')['profit'].sum().idxmin()

print("=" * 48)
print("         KEY BUSINESS INSIGHTS")
print("=" * 48)
print(f"  Total Revenue      : ${total_sales:>13,.2f}")
print(f"  Total Profit       : ${total_profit:>13,.2f}")
print(f"  Profit Margin      : {margin:>12.2f}%")
print(f"  Total Orders       : {total_orders:>13,}")
print(f"  Total Customers    : {total_customers:>13,}")
print(f"  Top Product        : {best_product}")
print(f"  Most Profitable Rgn: {best_region}")
print(f"  Best Segment       : {best_segment}")
print(f"  Loss-Making SubCat : {worst_subcat}")
print("=" * 48)